In [1]:
!pip install iterative-stratification

In [2]:
import os
import json
import numpy as np
import pandas as pd

import shutil

import matplotlib.pyplot as plt

import torch
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit
from sklearn.metrics import (
    precision_recall_fscore_support,
    accuracy_score,
    f1_score
)
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding
)

In [3]:
import sys
import importlib.metadata

print(f"Python: {sys.version.split()[0]}")

packages = [
    "torch",
    "transformers",
    "scikit-learn",
    "pandas",
    "numpy",
    "iterative-stratification"
]

for pkg in packages:
    try:
        print(f"{pkg}: {importlib.metadata.version(pkg)}")
    except importlib.metadata.PackageNotFoundError:
        print(f"{pkg}: Not installed")

Python: 3.13.15
torch: 2.11.0+cu128
transformers: 5.16.1
scikit-learn: 1.6.1
pandas: 2.2.3
numpy: 2.1.3
iterative-stratification: 0.1.9


In [4]:
SOURCE_FILE_NAME = "annotations.json"
FILTERED_FILE_NAME = "annotations_filtered.json"

ACD_EPOCHS_CSV = "acd_epochs.csv"
ACSA_EPOCHS_CSV = "acsa_epochs.csv"

# Diagram file paths
VIS_DATA_FILE_NAME = "token_lengths.json"
TOKEN_LENGTH_DISTRIBUTION_DIAGRAM_FILE_NAME = "token_length_distribution.png"

ACD_METRICS_PER_EPOCH = "acd_metrics_curves.png"
ACSA_METRICS_PER_EPOCH = "acsa_metrics_curves.png"

# Configurable Model Identifier: "classla/bcms-bertic" or "xlm-roberta-base"
MODEL_NAME = "classla/bcms-bertic"
SEED = 42
MAX_LEN = 512
EPOCHS = 10
SAVE_EPOCHS = 2
LR = 2e-5
WARMUP_STEPS = 0.1
WEIGHT_DECAY = 0.01
TRAIN_BATCH_SIZE = 16
EVAL_BATCH_SIZE = 16


# Paths to save trained models
SAVE_DIR = "saved_models"
SAVE_DIR_ACD = f"./{SAVE_DIR}/acd"
SAVE_DIR_ACSA = f"./{SAVE_DIR}/acsa"

ALL_CATEGORIES = [
    "Baterija", "Kamera", "Ekran", "Memorija", "Zvučnici",
    "Izgled", "Hardver", "Softver", "Performanse", "Cena", "Opšta ocena"
]

POLARITIES_MAP = {
    "Neutralan": 0, "Pozitivan": 1, "Negativan": 2, "Konflikt": 3
}

INV_POLARITIES_MAP = {
    val: key for key, val in POLARITIES_MAP.items()
}

In [5]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

with open(SOURCE_FILE_NAME, "r", encoding="utf-8") as f:
    data = json.load(f)

total_reviews = 0
over_512_count = 0
max_tokens_found = 0
all_lengths = []

print("Starting comment analysis...\n")

for item in data:
    text = item["comment"]
    review_status = item["review_status"]

    if review_status == "NE":
        continue

    total_reviews += 1

    token_ids = tokenizer.encode(text, add_special_tokens=True, truncation=False)
    num_tokens = len(token_ids)

    all_lengths.append(num_tokens)

    if num_tokens > 512:
        over_512_count += 1
        item["review_status"] = "NE"

    if num_tokens > max_tokens_found:
        max_tokens_found = num_tokens

percent = (over_512_count / total_reviews) * 100
mean_length = sum(all_lengths) / total_reviews

print("=== RESULT OF ANALYSIS OF REVIEWS TOKEN LENGTH ===")
print(f"Total number of reviews: {total_reviews}")
print(f"Number of reviews that go OVER 512 tokens: {over_512_count} ({percent:.2f}%)")
print(f"Mean review token length: {mean_length:.1f} tokena")
print(f"Longest reviews has: {max_tokens_found} tokens")

with open(FILTERED_FILE_NAME, "w", encoding="utf-8") as f:
    json.dump(data, f, indent=4, ensure_ascii=False)

viz_data = {
    "total_reviews": total_reviews,
    "over_512_count": over_512_count,
    "mean_length": mean_length,
    "max_length": max_tokens_found,
    "all_lengths": all_lengths
}

with open(VIS_DATA_FILE_NAME, "w", encoding="utf-8") as f:
    json.dump(viz_data, f, indent=4, ensure_ascii=False)

print(f"\nSuccessfully saved visualization data to '{VIS_DATA_FILE_NAME}'.")

plt.figure(figsize=(10, 6))
plt.hist(all_lengths, bins=40, color='skyblue', edgecolor='black', alpha=0.7)
plt.axvline(x=512, color='red', linestyle='--', linewidth=2, label='Token limit (512)')

plt.title('Distribution of token length in reviews', fontsize=14)
plt.xlabel('No. tokens', fontsize=12)
plt.ylabel('No. reviews', fontsize=12)
plt.legend(fontsize=11)
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()

plt.savefig(TOKEN_LENGTH_DISTRIBUTION_DIAGRAM_FILE_NAME, dpi=300)
plt.close()

print(f"Successfully saved diagram to '{TOKEN_LENGTH_DISTRIBUTION_DIAGRAM_FILE_NAME}'.")

config.json:   0%|          | 0.00/467 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/83.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/231k [00:00<?, ?B/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (1165 > 512). Running this sequence through the model will result in indexing errors


Starting comment analysis...

=== RESULT OF ANALYSIS OF REVIEWS TOKEN LENGTH ===
Total number of reviews: 6454
Number of reviews that go OVER 512 tokens: 11 (0.17%)
Mean review token length: 66.2 tokena
Longest reviews has: 1177 tokens

Successfully saved visualization data to 'token_lengths.json'.
Successfully saved diagram to 'token_length_distribution.png'.


In [6]:
# ===========================================
# 1. DATA PREPARATION (Document-Level Splitting)
# ===========================================
def load_and_split_data(json_file_path, seed=SEED):
    with open(json_file_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    # Filter unreviewed / invalid reviews
    valid_data = [item for item in data if item.get("review_status") != "NE"]

    # Build Category::Polarity multi-label matrix across all combinations
    pair_labels = [
        f"{cat}::{pol}"
        for cat in ALL_CATEGORIES
        for pol in POLARITIES_MAP.keys()
    ]
    pair_to_idx = {pair: idx for idx, pair in enumerate(pair_labels)}

    matrix = np.zeros((len(valid_data), len(pair_labels)), dtype=np.uint8)
    for i, item in enumerate(valid_data):
        for aspect_category in item.get("aspect_categories", []):
            cat = aspect_category.get("category")
            pol = aspect_category.get("polarity")
            pair = f"{cat}::{pol}"
            if pair in pair_to_idx:
                matrix[i, pair_to_idx[pair]] = 1

    # Stage 1 Split: 80% Train, 20% Held-out
    split1 = MultilabelStratifiedShuffleSplit(
        n_splits=1, test_size=0.20, random_state=seed
    )
    dummy_x = np.zeros((len(valid_data), 1), dtype=np.uint8)
    train_indices, held_out_indices = next(split1.split(dummy_x, matrix))

    # Stage 2 Split: Split Held-out 50/50 into Val and Test (10% and 10% global)
    split2 = MultilabelStratifiedShuffleSplit(
        n_splits=1, test_size=0.50, random_state=seed + 1
    )
    held_out_matrix = matrix[held_out_indices]
    dummy_held_out_x = np.zeros((len(held_out_matrix), 1), dtype=np.uint8)
    val_local, test_local = next(split2.split(dummy_held_out_x, held_out_matrix))

    val_indices = held_out_indices[val_local]
    test_indices = held_out_indices[test_local]

    train_items = [valid_data[i] for i in train_indices]
    val_items = [valid_data[i] for i in val_indices]
    test_items = [valid_data[i] for i in test_indices]

    return train_items, val_items, test_items

def extract_absa_samples(items):
    acd_comments, acd_labels = [], []
    acsa_comments, acsa_categories, acsa_labels = [], [], []
    gold_tuples = []  # Set of (category, polarity_id) per document

    for item in items:
        comment = item["comment"]
        aspect_categories = item.get("aspect_categories", [])

        # ACD binary indicator vector
        acd_comments.append(comment)
        label_vector = [0.0] * len(ALL_CATEGORIES)
        doc_tuples = set()

        for aspect_category in aspect_categories:
            category = aspect_category.get("category")
            polarity = aspect_category.get("polarity")

            if category in ALL_CATEGORIES:
                idx = ALL_CATEGORIES.index(category)
                label_vector[idx] = 1.0

                # ACSA Comment + Category pair
                if polarity in POLARITIES_MAP:
                    pol_id = POLARITIES_MAP[polarity]
                    acsa_comments.append(comment)
                    acsa_categories.append(category)
                    acsa_labels.append(pol_id)
                    doc_tuples.add((category, pol_id))

        acd_labels.append(label_vector)
        gold_tuples.append(doc_tuples)

    return (acd_comments, acd_labels), (acsa_comments, acsa_categories, acsa_labels), gold_tuples

def print_split_statistics(train_items, val_items, test_items):
    splits = {"Train": train_items, "Val": val_items, "Test": test_items}
    total_docs = sum(len(items) for items in splits.values())

    print("\n" + "=" * 70)
    print(" 1. DOCUMENT-LEVEL SPLIT RATIOS")
    print("=" * 70)
    for name, items in splits.items():
        count = len(items)
        pct = (count / total_docs) * 100
        print(f" {name:<10}: {count:>5} comments ({pct:>6.2f}%)")

    # Aggregate counts across splits
    pair_labels = [
        f"{cat}::{pol}"
        for cat in ALL_CATEGORIES
        for pol in POLARITIES_MAP.keys()
    ]
    pair_stats = {name: {pair: 0 for pair in pair_labels} for name in splits}
    cat_stats = {name: {cat: 0 for cat in ALL_CATEGORIES} for name in splits}

    for name, items in splits.items():
        for item in items:
            for aspect in item.get("aspect_categories", []):
                cat = aspect.get("category")
                pol = aspect.get("polarity")
                pair = f"{cat}::{pol}"
                if pair in pair_stats[name]:
                    pair_stats[name][pair] += 1
                if cat in cat_stats[name]:
                    cat_stats[name][cat] += 1

    print("\n" + "=" * 70)
    print(" 2. CATEGORY-LEVEL DISTRIBUTION (ACD)")
    print(" Target split ratio: Train (~80%) | Val (~10%) | Test (~10%)")
    print("=" * 70)
    print(f" {'Category':<15} | {'Total':<6} | {'Train %':<8} | {'Val %':<8} | {'Test %':<8}")
    print("-" * 70)
    for cat in ALL_CATEGORIES:
        tr_c = cat_stats["Train"][cat]
        va_c = cat_stats["Val"][cat]
        te_c = cat_stats["Test"][cat]
        tot = tr_c + va_c + te_c
        if tot > 0:
            print(f" {cat:<15} | {tot:<6} | {tr_c/tot:>7.1%} | {va_c/tot:>7.1%} | {te_c/tot:>7.1%}")
        else:
            print(f" {cat:<15} | {tot:<6} | {'N/A':>7} | {'N/A':>7} | {'N/A':>7}")

    print("\n" + "=" * 70)
    print(" 3. CATEGORY::POLARITY PAIR DISTRIBUTION (ACSA)")
    print("=" * 70)
    print(f" {'Pair Label':<30} | {'Total':<6} | {'Train %':<8} | {'Val %':<8} | {'Test %':<8}")
    print("-" * 70)

    # Sort pairs by frequency
    sorted_pairs = sorted(
        pair_labels,
        key=lambda p: sum(pair_stats[n][p] for n in splits),
        reverse=True
    )

    for pair in sorted_pairs:
        tr_c = pair_stats["Train"][pair]
        va_c = pair_stats["Val"][pair]
        te_c = pair_stats["Test"][pair]
        tot = tr_c + va_c + te_c
        if tot > 0:
            print(f" {pair:<30} | {tot:<6} | {tr_c/tot:>7.1%} | {va_c/tot:>7.1%} | {te_c/tot:>7.1%}")
    print("=" * 70 + "\n")

In [7]:
# ===========================================
# 2. LAZY DATASETS (Dynamic Padding)
# ===========================================
class LazyACDDataset(torch.utils.data.Dataset):
    def __init__(self, comments, labels, tokenizer, max_len=MAX_LEN):
        self.comments = comments
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __getitem__(self, idx):
        item = self.tokenizer(
            self.comments[idx],
            truncation=True,
            max_length=self.max_len
        )
        item["labels"] = self.labels[idx]
        return item

    def __len__(self):
        return len(self.labels)

class LazyACSADataset(torch.utils.data.Dataset):
    def __init__(self, comments, categories, labels, tokenizer, max_len=MAX_LEN):
        self.comments = comments
        self.categories = categories
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __getitem__(self, idx):
        item = self.tokenizer(
            self.comments[idx],
            text_pair=self.categories[idx],
            truncation=True,
            max_length=self.max_len
        )
        item["labels"] = self.labels[idx]
        return item

    def __len__(self):
        return len(self.labels)

In [8]:
# ===========================================
# 3. METRICS & THRESHOLD OPTIMIZATION
# ===========================================
def get_compute_metrics_acd(threshold=0.5):
    def compute_metrics_acd(eval_pred):
        logits, labels = eval_pred
        probs = 1 / (1 + np.exp(-logits))
        predictions = (probs > threshold).astype(int)
        acc = accuracy_score(labels, predictions)
        macro_precision, macro_recall, macro_f1, _ = precision_recall_fscore_support(labels, predictions, average="macro", zero_division=0)
        micro_f1 = f1_score(labels, predictions, average="micro", zero_division=0)
        weighted_f1 = f1_score(labels, predictions, average="weighted", zero_division=0)
        return {
            "acd_accuracy": acc,
            "acd_macro_f1": macro_f1,
            "acd_macro_precision": macro_precision,
            "acd_macro_recall": macro_recall,
            "acd_micro_f1": micro_f1,
            "acd_weighted_f1": weighted_f1
        }
    return compute_metrics_acd

def compute_metrics_acsa(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, predictions)
    macro_precision, macro_recall, macro_f1, _ = precision_recall_fscore_support(labels, predictions, average="macro", zero_division=0)
    micro_f1 = f1_score(labels, predictions, average="micro", zero_division=0)
    weighted_f1 = f1_score(labels, predictions, average="weighted", zero_division=0)
    return {
        "acsa_accuracy": acc,
        "acsa_macro_f1": macro_f1,
        "acsa_macro_precision": macro_precision,
        "acsa_macro_recall": macro_recall,
        "acsa_micro_f1": micro_f1,
        "acsa_weighted_f1": weighted_f1
    }

def find_best_acd_thresholds_per_class(
    model, val_dataset, data_collator, device, categories
):
    model.to(device).eval()
    val_loader = torch.utils.data.DataLoader(
        val_dataset, batch_size=16, collate_fn=data_collator
    )

    all_logits, all_labels = [], []
    with torch.no_grad():
        for batch in val_loader:
            labels = batch.pop("labels")
            inputs = {k: v.to(device) for k, v in batch.items()}
            logits = model(**inputs).logits
            all_logits.append(logits.cpu().numpy())
            all_labels.append(labels.numpy())

    all_logits = np.vstack(all_logits)
    all_labels = np.vstack(all_labels)
    probs = 1 / (1 + np.exp(-all_logits))

    best_thresholds = {}
    grid = np.arange(0.05, 0.90, 0.05)

    print("\n[Validation] Optimizing Per-Class ACD Thresholds:")
    for idx, cat in enumerate(categories):
        cat_probs = probs[:, idx]
        cat_labels = all_labels[:, idx]

        best_t = 0.5
        best_f1 = -1.0

        for thresh in grid:
            preds = (cat_probs > thresh).astype(int)
            f1 = f1_score(cat_labels, preds, average="binary", zero_division=0)
            if f1 > best_f1:
                best_f1 = f1
                best_t = thresh

        best_thresholds[cat] = float(best_t)
        print(
            f" - {cat:<15}: Threshold = {best_t:.2f} (Val F1: {best_f1:.4f})"
        )

    return best_thresholds

def evaluate_acd_overall(
    model, dataset, data_collator, thresholds, categories, device
):
    model.to(device).eval()
    loader = torch.utils.data.DataLoader(
        dataset, batch_size=16, collate_fn=data_collator
    )

    all_logits, all_labels = [], []
    with torch.no_grad():
        for batch in loader:
            labels = batch.pop("labels")
            inputs = {k: v.to(device) for k, v in batch.items()}
            logits = model(**inputs).logits
            all_logits.append(logits.cpu().numpy())
            all_labels.append(labels.numpy())

    all_logits = np.vstack(all_logits)
    all_labels = np.vstack(all_labels)
    probs = 1 / (1 + np.exp(-all_logits))

    if isinstance(thresholds, dict):
        thresh_arr = np.array([thresholds[cat] for cat in categories])
    else:
        thresh_arr = np.array(thresholds)

    predictions = (probs > thresh_arr).astype(int)

    acc = accuracy_score(all_labels, predictions)
    macro_precision, macro_recall, macro_f1, _ = (
        precision_recall_fscore_support(
            all_labels, predictions, average="macro", zero_division=0
        )
    )
    micro_f1 = f1_score(
        all_labels, predictions, average="micro", zero_division=0
    )
    weighted_f1 = f1_score(
        all_labels, predictions, average="weighted", zero_division=0
    )

    precisions, recalls, f1s, supports = precision_recall_fscore_support(
        all_labels, predictions, average=None, zero_division=0
    )

    category_metrics = []
    for idx, cat in enumerate(categories):
        category_metrics.append(
            {
                "category": cat,
                "threshold": float(thresh_arr[idx]),
                "f1": f1s[idx],
                "precision": precisions[idx],
                "recall": recalls[idx],
                "support": int(supports[idx]),
            }
        )

    category_metrics.sort(key=lambda x: x["f1"], reverse=True)

    return {
        "acd_accuracy": acc,
        "acd_macro_f1": macro_f1,
        "acd_macro_precision": macro_precision,
        "acd_macro_recall": macro_recall,
        "acd_micro_f1": micro_f1,
        "acd_weighted_f1": weighted_f1,
        "category_metrics": category_metrics
    }

In [9]:
# ===========================================
# 4. END-TO-END PIPELINE EVALUATION
# ===========================================
def evaluate_end_to_end(
    acd_model, acd_test_ds, acsa_model, tokenizer, test_comments, gold_tuples_list, acd_thresholds, batch_size=16
):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    acd_model.to(device).eval()
    acsa_model.to(device).eval()

    if isinstance(acd_thresholds, dict):
        thresh_arr = np.array([acd_thresholds[cat] for cat in ALL_CATEGORIES])
    else:
        thresh_arr = np.array(acd_thresholds)

    # Run ACD inference in batches
    acd_loader = torch.utils.data.DataLoader(
        acd_test_ds, batch_size=batch_size, collate_fn=DataCollatorWithPadding(tokenizer)
    )
    all_acd_probs = []
    with torch.no_grad():
        for batch in acd_loader:
            batch.pop("labels", None)
            inputs = {k: v.to(device) for k, v in batch.items()}
            logits = acd_model(**inputs).logits
            all_acd_probs.append(torch.sigmoid(logits).cpu().numpy())
    all_acd_probs = np.vstack(all_acd_probs)

    # Collect positive pairs & track indexing
    acsa_inputs, pair_indices = [], []
    y_true, y_pred = [], []
    active_labels = [0, 1, 2, 3]
    pred_tuples_list = [set() for _ in range(len(test_comments))]

    for doc_idx, (comment, gold_tuples) in enumerate(zip(test_comments, gold_tuples_list)):
        gold_dict = dict(gold_tuples)
        acd_probs = all_acd_probs[doc_idx]

        for cat_idx, cat in enumerate(ALL_CATEGORIES):
            gold_pol = gold_dict.get(cat, -1)
            y_true.append(gold_pol)
            curr_flat_idx = len(y_true) - 1

            if acd_probs[cat_idx] > thresh_arr[cat_idx]:
                acsa_inputs.append((comment, cat))
                pair_indices.append(curr_flat_idx)
                y_pred.append(-1)
            else:
                y_pred.append(-1)

    # Run ACSA inference
    if acsa_inputs:
        acsa_preds = []
        for i in range(0, len(acsa_inputs), batch_size):
            batch_pairs = acsa_inputs[i:i + batch_size]
            encoded = tokenizer(
                [p[0] for p in batch_pairs],
                [p[1] for p in batch_pairs],
                padding=True,
                truncation=True,
                max_length=MAX_LEN,
                return_tensors="pt"
            ).to(device)
            with torch.no_grad():
                logits = acsa_model(**encoded).logits
                acsa_preds.extend(torch.argmax(logits, dim=-1).cpu().tolist())

        for idx, pred_pol in zip(pair_indices, acsa_preds):
            y_pred[idx] = pred_pol

    # Populate predicted tuple sets
    for flat_idx, pred_pol in enumerate(y_pred):
        if pred_pol in active_labels:
            doc_idx = flat_idx // len(ALL_CATEGORIES)
            cat = ALL_CATEGORIES[flat_idx % len(ALL_CATEGORIES)]
            pred_tuples_list[doc_idx].add((cat, pred_pol))

    # --- Flat-Level Metrics ---
    macro_precision, macro_recall, macro_f1, _ = precision_recall_fscore_support(
        y_true, y_pred, labels=active_labels, average="macro", zero_division=0
    )
    micro_precision, micro_recall, micro_f1, _ = precision_recall_fscore_support(
        y_true, y_pred, labels=active_labels, average="micro", zero_division=0
    )
    weighted_f1 = f1_score(
        y_true, y_pred, labels=active_labels, average="weighted", zero_division=0
    )

    # --- Tuple-Level Metrics ---
    total_tp, total_fp, total_fn = 0, 0, 0
    doc_f1s = []

    for gold_tuples, pred_tuples in zip(gold_tuples_list, pred_tuples_list):
        tp = len(gold_tuples & pred_tuples)
        fp = len(pred_tuples - gold_tuples)
        fn = len(gold_tuples - pred_tuples)

        total_tp += tp
        total_fp += fp
        total_fn += fn

        prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        rec = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        doc_f1 = (2 * prec * rec) / (prec + rec) if (prec + rec) > 0 else 0.0
        doc_f1s.append(doc_f1)

    tuple_micro_prec = total_tp / (total_tp + total_fp) if (total_tp + total_fp) > 0 else 0.0
    tuple_micro_rec = total_tp / (total_tp + total_fn) if (total_tp + total_fn) > 0 else 0.0
    tuple_micro_f1 = (2 * tuple_micro_prec * tuple_micro_rec) / (tuple_micro_prec + tuple_micro_rec) if (tuple_micro_prec + tuple_micro_rec) > 0 else 0.0

    return {
        "e2e_macro_f1": macro_f1,
        "e2e_macro_precision": macro_precision,
        "e2e_macro_recall": macro_recall,
        "e2e_micro_f1": micro_f1,
        "e2e_micro_precision": micro_precision,
        "e2e_micro_recall": micro_recall,
        "e2e_weighted_f1": weighted_f1,
        # tuple metrics
        "tuple_macro_f1": float(np.mean(doc_f1s)),
        "tuple_micro_f1": tuple_micro_f1,
        "tuple_micro_precision": tuple_micro_prec,
        "tuple_micro_recall": tuple_micro_rec,
    }

In [10]:
# ===========================================
# 5. SAVE PER EPOCH TRAINING METRICS
# ===========================================
def save_epoch_metrics_to_csv(log_history, csv_filename, task_type="acd"):
    eval_logs = [entry for entry in log_history if "eval_loss" in entry]
    train_logs = [entry for entry in log_history if "loss" in entry]

    rows = []
    for eval_entry in eval_logs:
        epoch_num = int(round(eval_entry["epoch"]))

        # Pick the latest logged training step loss at or before this evaluation
        matching_train = [t for t in train_logs if t["epoch"] <= eval_entry["epoch"]]
        train_loss = matching_train[-1]["loss"] if matching_train else eval_entry.get("loss", np.nan)

        prefix = task_type.lower()
        prefix_cap = prefix.capitalize()

        row = {
            "Epoch": epoch_num,
            "Training Loss": train_loss,
            "Validation Loss": eval_entry.get("eval_loss", np.nan),
            f"{prefix_cap} Accuracy": eval_entry.get(f"eval_{prefix}_accuracy", np.nan),
            f"{prefix_cap} Macro F1": eval_entry.get(f"eval_{prefix}_macro_f1", np.nan),
            f"{prefix_cap} Macro Precision": eval_entry.get(f"eval_{prefix}_macro_precision", np.nan),
            f"{prefix_cap} Macro Recall": eval_entry.get(f"eval_{prefix}_macro_recall", np.nan),
            f"{prefix_cap} Micro F1": eval_entry.get(f"eval_{prefix}_micro_f1", np.nan),
            f"{prefix_cap} Weighted F1": eval_entry.get(f"eval_{prefix}_weighted_f1", np.nan),
        }
        rows.append(row)

    df = pd.DataFrame(rows)
    # Deduplicate evaluations for the same epoch, keeping the first (in-training) evaluation
    df = df.drop_duplicates(subset=["Epoch"], keep="first")

    df.to_csv(csv_filename, index=False, float_format="%.6f")
    print(f"Successfully generated and saved {csv_filename}")

In [11]:
# ===========================================
# 6. MAIN EXECUTION FLOW
# ===========================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# 1. Load and split data
train_items, val_items, test_items = load_and_split_data(FILTERED_FILE_NAME)

(acd_train_c, acd_train_l), (acsa_train_c, acsa_train_cat, acsa_train_l), _ = extract_absa_samples(train_items)
(acd_val_c, acd_val_l), (acsa_val_c, acsa_val_cat, acsa_val_l), _ = extract_absa_samples(val_items)
(acd_test_c, acd_test_l), (acsa_test_c, acsa_test_cat, acsa_test_l), gold_test_tuples = extract_absa_samples(test_items)

# 2. Build Datasets
acd_train_ds = LazyACDDataset(acd_train_c, acd_train_l, tokenizer)
acd_val_ds = LazyACDDataset(acd_val_c, acd_val_l, tokenizer)
acd_test_ds = LazyACDDataset(acd_test_c, acd_test_l, tokenizer)

acsa_train_ds = LazyACSADataset(acsa_train_c, acsa_train_cat, acsa_train_l, tokenizer)
acsa_val_ds = LazyACSADataset(acsa_val_c, acsa_val_cat, acsa_val_l, tokenizer)
acsa_test_ds = LazyACSADataset(acsa_test_c, acsa_test_cat, acsa_test_l, tokenizer)

In [12]:
print_split_statistics(train_items, val_items, test_items)


 1. DOCUMENT-LEVEL SPLIT RATIOS
 Train     :  5153 comments ( 79.98%)
 Val       :   638 comments (  9.90%)
 Test      :   652 comments ( 10.12%)

 2. CATEGORY-LEVEL DISTRIBUTION (ACD)
 Target split ratio: Train (~80%) | Val (~10%) | Test (~10%)
 Category        | Total  | Train %  | Val %    | Test %  
----------------------------------------------------------------------
 Baterija        | 2191   |   80.0% |    9.9% |   10.0%
 Kamera          | 1429   |   80.0% |    9.9% |   10.1%
 Ekran           | 961    |   80.0% |   10.0% |   10.0%
 Memorija        | 164    |   80.5% |    9.8% |    9.8%
 Zvučnici        | 500    |   80.0% |   10.0% |   10.0%
 Izgled          | 839    |   80.0% |   10.0% |   10.0%
 Hardver         | 1387   |   80.0% |   10.0% |    9.9%
 Softver         | 1841   |   80.0% |   10.0% |   10.0%
 Performanse     | 1126   |   79.9% |    9.9% |   10.1%
 Cena            | 853    |   80.0% |   10.2% |    9.8%
 Opšta ocena     | 3091   |   80.0% |   10.0% |   10.0%

 3. CA

In [13]:
# --- STAGE 1: TRAIN ACD ---
acd_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(ALL_CATEGORIES),
    problem_type="multi_label_classification"
)
acd_args = TrainingArguments(
    output_dir="./results_acd",
    num_train_epochs=EPOCHS,
    save_total_limit=SAVE_EPOCHS,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    learning_rate=LR,
    warmup_steps=WARMUP_STEPS,
    weight_decay=WEIGHT_DECAY,
    fp16=torch.cuda.is_available(),
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="acd_macro_f1",
    greater_is_better=True,
    logging_steps=50
)
acd_trainer = Trainer(
    model=acd_model,
    args=acd_args,
    train_dataset=acd_train_ds,
    eval_dataset=acd_val_ds,
    compute_metrics=get_compute_metrics_acd(0.5),
    data_collator=data_collator,
    processing_class=tokenizer
)

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  443MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] ElectraForSequenceClassification LOAD REPORT from: classla/bcms-bertic
Key                                               | Status     | 
--------------------------------------------------+------------+-
discriminator_predictions.dense_prediction.weight | UNEXPECTED | 
discriminator_predictions.dense.bias              | UNEXPECTED | 
discriminator_predictions.dense.weight            | UNEXPECTED | 
discriminator_predictions.dense_prediction.bias   | UNEXPECTED | 
classifier.dense.weight                           | MISSING    | 
classifier.out_proj.weight                        | MISSING    | 
classifier.dense.bias                             | MISSING    | 
classifier.out_proj.bias                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [14]:
print("--- Starting Training for Stage 1: ACD ---")
acd_trainer.train()

--- Starting Training for Stage 1: ACD ---


Epoch,Training Loss,Validation Loss,Acd Accuracy,Acd Macro F1,Acd Macro Precision,Acd Macro Recall,Acd Micro F1,Acd Weighted F1
1,0.426280,0.391313,0.026646,0.219505,0.441271,0.173175,0.413758,0.340586
2,0.321890,0.312670,0.288401,0.495148,0.626006,0.452294,0.665382,0.621242
3,0.259609,0.254338,0.366771,0.589911,0.679537,0.536038,0.729239,0.708617
4,0.212737,0.224016,0.421630,0.605798,0.728795,0.540608,0.749612,0.727910
5,0.168279,0.205208,0.451411,0.656413,0.723994,0.614024,0.780398,0.767609
6,0.149738,0.192220,0.509404,0.714835,0.730864,0.705012,0.820154,0.814789
7,0.124228,0.183909,0.515674,0.722555,0.757271,0.694199,0.825135,0.817612
8,0.108242,0.180932,0.525078,0.727328,0.748591,0.713450,0.829165,0.823918
9,0.101485,0.179631,0.534483,0.733955,0.754954,0.717959,0.833865,0.828250
10,0.098179,0.178009,0.537618,0.738097,0.743971,0.735474,0.835134,0.830580


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=3230, training_loss=0.21784474325622938, metrics={'train_runtime': 715.3905, 'train_samples_per_second': 72.031, 'train_steps_per_second': 4.515, 'total_flos': 5522510567296146.0, 'train_loss': 0.21784474325622938, 'epoch': 10.0})

In [15]:
#log_df = pd.DataFrame(acd_trainer.state.log_history)
#print(log_df.head())

# Export Stage 1 metrics to CSV
save_epoch_metrics_to_csv(
    acd_trainer.state.log_history,
    ACD_EPOCHS_CSV,
    task_type="acd")

Successfully generated and saved acd_epochs.csv


In [16]:
# Find optimal per-class probability thresholds on validation set
best_acd_thresholds = find_best_acd_thresholds_per_class(
    acd_model, acd_val_ds, data_collator, device, ALL_CATEGORIES
)


[Validation] Optimizing Per-Class ACD Thresholds:
 - Baterija       : Threshold = 0.30 (Val F1: 0.9615)
 - Kamera         : Threshold = 0.60 (Val F1: 0.9071)
 - Ekran          : Threshold = 0.40 (Val F1: 0.7081)
 - Memorija       : Threshold = 0.15 (Val F1: 0.3000)
 - Zvučnici       : Threshold = 0.35 (Val F1: 0.7959)
 - Izgled         : Threshold = 0.40 (Val F1: 0.7113)
 - Hardver        : Threshold = 0.40 (Val F1: 0.7875)
 - Softver        : Threshold = 0.55 (Val F1: 0.8385)
 - Performanse    : Threshold = 0.25 (Val F1: 0.7833)
 - Cena           : Threshold = 0.50 (Val F1: 0.8506)
 - Opšta ocena    : Threshold = 0.60 (Val F1: 0.8918)


In [17]:
# Overall and per-category ACD evaluation on Test Set using per-class thresholds
acd_results = evaluate_acd_overall(
    acd_model,
    acd_test_ds,
    data_collator,
    best_acd_thresholds,
    ALL_CATEGORIES,
    device,
)
print("\n--- Standalone Overall Evaluation for ACD (Test Set) ---")
for metric, val in acd_results.items():
    if metric != "category_metrics":
        print(f"  {metric:<20}: {val:.4f}")

print("\n--- ACD Per-Category Evaluation (Test Set, Sorted by F1) ---")
print(
    f"{'Category':<15} | {'Thresh':<6} | {'F1 Score':<10} | {'Precision':<10} |"
    " {'Recall':<10} | {'Support':<8}"
)
print("-" * 75)
for item in acd_results["category_metrics"]:
    print(
        f"{item['category']:<15} | {item['threshold']:<6.2f} |"
        f" {item['f1']:<10.4f} | {item['precision']:<10.4f} |"
        f" {item['recall']:<10.4f} | {item['support']:<8}"
    )


--- Standalone Overall Evaluation for ACD (Test Set) ---
  acd_accuracy        : 0.5261
  acd_macro_f1        : 0.7729
  acd_macro_precision : 0.7538
  acd_macro_recall    : 0.8008
  acd_micro_f1        : 0.8251
  acd_weighted_f1     : 0.8275

--- ACD Per-Category Evaluation (Test Set, Sorted by F1) ---
Category        | Thresh | F1 Score   | Precision  | {'Recall':<10} | {'Support':<8}
---------------------------------------------------------------------------
Baterija        | 0.30   | 0.9522     | 0.9125     | 0.9955     | 220     
Kamera          | 0.60   | 0.9172     | 0.9110     | 0.9236     | 144     
Zvučnici        | 0.35   | 0.8800     | 0.8800     | 0.8800     | 50      
Opšta ocena     | 0.60   | 0.8524     | 0.8712     | 0.8344     | 308     
Hardver         | 0.40   | 0.8043     | 0.8043     | 0.8043     | 138     
Cena            | 0.50   | 0.8000     | 0.8421     | 0.7619     | 84      
Softver         | 0.55   | 0.7718     | 0.8012     | 0.7446     | 184     
Ekran   

In [18]:
# --- SAVE STAGE 1 (ACD MODEL & CONFIG) ---
print(f"\n[Saving] Saving Stage 1 (ACD) model to {SAVE_DIR_ACD}...")
acd_trainer.save_model(SAVE_DIR_ACD)
tokenizer.save_pretrained(SAVE_DIR_ACD)

acd_config = {
    "best_thresholds": best_acd_thresholds,
    "categories": ALL_CATEGORIES,
}
with open(os.path.join(SAVE_DIR_ACD, "acd_config.json"), "w", encoding="utf-8") as f:
    json.dump(acd_config, f, ensure_ascii=False, indent=2)


[Saving] Saving Stage 1 (ACD) model to ./saved_models/acd...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [19]:
# --- STAGE 2: TRAIN ACSA ---
acsa_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(POLARITIES_MAP)
)
acsa_args = TrainingArguments(
    output_dir="./results_acsa",
    num_train_epochs=EPOCHS,
    save_total_limit=SAVE_EPOCHS,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    learning_rate=LR,
    warmup_steps=WARMUP_STEPS,
    weight_decay=WEIGHT_DECAY,
    fp16=torch.cuda.is_available(),
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="acsa_macro_f1",
    greater_is_better=True,
    logging_steps=50
)
acsa_trainer = Trainer(
    model=acsa_model,
    args=acsa_args,
    train_dataset=acsa_train_ds,
    eval_dataset=acsa_val_ds,
    compute_metrics=compute_metrics_acsa,
    data_collator=data_collator,
    processing_class=tokenizer
)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] ElectraForSequenceClassification LOAD REPORT from: classla/bcms-bertic
Key                                               | Status     | 
--------------------------------------------------+------------+-
discriminator_predictions.dense_prediction.weight | UNEXPECTED | 
discriminator_predictions.dense.bias              | UNEXPECTED | 
discriminator_predictions.dense.weight            | UNEXPECTED | 
discriminator_predictions.dense_prediction.bias   | UNEXPECTED | 
classifier.dense.weight                           | MISSING    | 
classifier.out_proj.weight                        | MISSING    | 
classifier.dense.bias                             | MISSING    | 
classifier.out_proj.bias                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [20]:
print("\n--- Starting Training for Stage 2: ACSA ---")
acsa_trainer.train()


--- Starting Training for Stage 2: ACSA ---


Epoch,Training Loss,Validation Loss,Acsa Accuracy,Acsa Macro F1,Acsa Macro Precision,Acsa Macro Recall,Acsa Micro F1,Acsa Weighted F1
1,0.518937,0.552603,0.824757,0.424663,0.415655,0.436569,0.824757,0.796066
2,0.368319,0.482437,0.863700,0.445862,0.432776,0.460149,0.863700,0.834383
3,0.343285,0.437508,0.874131,0.544302,0.672702,0.521057,0.874131,0.856607
4,0.264242,0.477498,0.867177,0.610424,0.609265,0.612583,0.867177,0.867641
5,0.240122,0.551034,0.873435,0.618803,0.640862,0.605631,0.873435,0.869775
6,0.191487,0.604577,0.861613,0.631238,0.615531,0.653064,0.861613,0.866659
7,0.140992,0.648040,0.881085,0.627158,0.645300,0.613104,0.881085,0.877335
8,0.079174,0.729794,0.866481,0.621673,0.616029,0.629821,0.866481,0.869229
9,0.098259,0.750370,0.865090,0.609956,0.601919,0.619951,0.865090,0.868115
10,0.038999,0.764100,0.869958,0.616350,0.608016,0.627001,0.869958,0.873069


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=7200, training_loss=0.2625981151560942, metrics={'train_runtime': 2197.4916, 'train_samples_per_second': 52.36, 'train_steps_per_second': 3.276, 'total_flos': 1.5865512561019056e+16, 'train_loss': 0.2625981151560942, 'epoch': 10.0})

In [21]:
# Export Stage 2 metrics to CSV
save_epoch_metrics_to_csv(
    acsa_trainer.state.log_history,
    ACSA_EPOCHS_CSV,
    task_type="acsa")

Successfully generated and saved acsa_epochs.csv


In [22]:
# Standalone ACSA (Oracle) Test Evaluation
print("\n--- Standalone Oracle Evaluation for ACSA (Test Set) ---")
acsa_test_results = acsa_trainer.evaluate(eval_dataset=acsa_test_ds)
print(acsa_test_results)


--- Standalone Oracle Evaluation for ACSA (Test Set) ---


Training Loss,Validation Loss,Epoch,Acsa Accuracy,Acsa Macro F1,Acsa Macro Precision,Acsa Macro Recall,Acsa Micro F1,Acsa Weighted F1
0.038999,0.533402,10,0.870654,0.612099,0.602008,0.625997,0.870654,0.875502


{'eval_loss': 0.5334018468856812, 'eval_acsa_accuracy': 0.8706536856745479, 'eval_acsa_macro_f1': 0.6120990549451591, 'eval_acsa_macro_precision': 0.602007606655568, 'eval_acsa_macro_recall': 0.6259971444304792, 'eval_acsa_micro_f1': 0.8706536856745479, 'eval_acsa_weighted_f1': 0.8755021978855296}


In [23]:
# --- SAVE STAGE 2 (ACSA MODEL & CONFIG) ---
print(f"\n[Saving] Saving Stage 2 (ACSA) model to {SAVE_DIR_ACSA}...")
acsa_trainer.save_model(SAVE_DIR_ACSA)
tokenizer.save_pretrained(SAVE_DIR_ACSA)

acsa_config = {
    "polarities_map": POLARITIES_MAP
}
with open(os.path.join(SAVE_DIR_ACSA, "acsa_config.json"), "w", encoding="utf-8") as f:
    json.dump(acsa_config, f, ensure_ascii=False, indent=2)


[Saving] Saving Stage 2 (ACSA) model to ./saved_models/acsa...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [24]:
# --- END-TO-END PIPELINE EVALUATION ---
print("\n--- End-to-End Pipeline Evaluation (Test Set) ---")
e2e_results = evaluate_end_to_end(
    acd_model=acd_model,
    acd_test_ds=acd_test_ds,
    acsa_model=acsa_model,
    tokenizer=tokenizer,
    test_comments=acd_test_c,
    gold_tuples_list=gold_test_tuples,
    acd_thresholds=best_acd_thresholds,
)
print(e2e_results)


--- End-to-End Pipeline Evaluation (Test Set) ---
{'e2e_macro_f1': 0.5233605123898016, 'e2e_macro_precision': 0.5013248607760803, 'e2e_macro_recall': 0.5511725171853287, 'e2e_micro_f1': 0.7247457627118644, 'e2e_micro_precision': 0.707010582010582, 'e2e_micro_recall': 0.7433936022253129, 'e2e_weighted_f1': 0.7283588874041957, 'tuple_macro_f1': 0.73893515762877, 'tuple_micro_f1': 0.7247457627118644, 'tuple_micro_precision': 0.707010582010582, 'tuple_micro_recall': 0.7433936022253129}


In [25]:
# Creates a f'{SAVE_DIR}.zip' file in your current directory
shutil.make_archive(SAVE_DIR, "zip", SAVE_DIR)

'/content/saved_models.zip'

In [26]:
df_acd = pd.read_csv(ACD_EPOCHS_CSV)
df_acsa = pd.read_csv(ACSA_EPOCHS_CSV)

# Setup plot style
plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")
epochs = range(1, 11)

# ---------------------------------------------------------
# DIAGRAM: STAGE 1 (ACD)
# ---------------------------------------------------------
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Plot ACD Loss
ax1.plot(epochs, df_acd["Training Loss"], marker="o", color="#1f77b4", linewidth=2, label="Train Loss")
ax1.plot(epochs, df_acd["Validation Loss"], marker="s", color="#d62728", linewidth=2, linestyle="--", label="Val Loss")
ax1.set_title("Stage 1 (ACD): Training & Validation Loss", fontsize=12, fontweight="bold")
ax1.set_xlabel("Epoch", fontsize=11)
ax1.set_ylabel("Loss", fontsize=11)
ax1.set_xticks(epochs)
ax1.legend(frameon=True)
ax1.grid(True, alpha=0.3)

# Plot ACD Metrics
ax2.plot(epochs, df_acd["Acd Macro F1"], marker="o", color="#2ca02c", linewidth=2, label="Macro F1")
ax2.plot(epochs, df_acd["Acd Micro F1"], marker="^", color="#ff7f0e", linewidth=2, label="Micro F1")
ax2.plot(epochs, df_acd["Acd Accuracy"], marker="d", color="#9467bd", linewidth=2, linestyle=":", label="Accuracy")
ax2.set_title("Stage 1 (ACD): Performance Metrics", fontsize=12, fontweight="bold")
ax2.set_xlabel("Epoch", fontsize=11)
ax2.set_ylabel("Score", fontsize=11)
ax2.set_xticks(epochs)
ax2.legend(frameon=True)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(ACD_METRICS_PER_EPOCH, dpi=300)
plt.close()
print(f"Successfully generated and saved {ACD_METRICS_PER_EPOCH}")

# ---------------------------------------------------------
# DIAGRAM: STAGE 2 (ACSA)
# ---------------------------------------------------------
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Plot ACSA Loss
ax1.plot(epochs, df_acsa["Training Loss"], marker="o", color="#1f77b4", linewidth=2, label="Train Loss")
ax1.plot(epochs, df_acsa["Validation Loss"], marker="s", color="#d62728", linewidth=2, linestyle="--", label="Val Loss")
ax1.set_title("Stage 2 (ACSA): Training & Validation Loss", fontsize=12, fontweight="bold")
ax1.set_xlabel("Epoch", fontsize=11)
ax1.set_ylabel("Loss", fontsize=11)
ax1.set_xticks(epochs)
ax1.legend(frameon=True)
ax1.grid(True, alpha=0.3)

# Plot ACSA Metrics
ax2.plot(epochs, df_acsa["Acsa Macro F1"], marker="o", color="#2ca02c", linewidth=2, label="Macro F1")
ax2.plot(epochs, df_acsa["Acsa Micro F1"], marker="^", color="#ff7f0e", linewidth=2, label="Micro F1")
ax2.plot(epochs, df_acsa["Acsa Accuracy"], marker="d", color="#9467bd", linewidth=2, linestyle=":", label="Accuracy")
ax2.set_title("Stage 2 (ACSA): Performance Metrics", fontsize=12, fontweight="bold")
ax2.set_xlabel("Epoch", fontsize=11)
ax2.set_ylabel("Score", fontsize=11)
ax2.set_xticks(epochs)
ax2.legend(frameon=True)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(ACSA_METRICS_PER_EPOCH, dpi=300)
plt.close()
print(f"Successfully generated and saved {ACSA_METRICS_PER_EPOCH}")

Successfully generated and saved acd_metrics_curves.png
Successfully generated and saved acsa_metrics_curves.png
